In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as w

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")

# 1. Engineer leak-free features available at order placement
featured_df = (
    orders_df
    .filter(F.col("promised_date").isNotNull())
    .withColumn("promised_days", F.datediff(F.col("promised_date"), F.col("order_date")))
    .withColumn("order_dow", F.dayofweek(F.col("order_date")))
    .withColumn("order_month", F.month(F.col("order_date")))
    .fillna({
        "quantity": 0,
        "amount": 0.0,
        "discount_pct": 0.0,
        "shipping_cost": 0.0,
        "sku_rating": 0,
        "promised_days": 0
    })
)

# 2. Split into historical delivered orders (for training) and open orders (for scoring)
historical_df = (
    featured_df
    .filter(F.col("delivered_date").isNotNull())
    .withColumn(
        "label",
        F.when(F.col("delivered_date") > F.col("promised_date"), 1.0).otherwise(0.0)
    )
)

open_orders_df = featured_df.filter(F.col("delivered_date").isNull())

# 3. Build preprocessing & Random Forest classification pipeline
cat_cols = ["warehouse_id", "delivery_partner_id", "sku_id"]
num_cols = ["promised_days", "order_dow", "order_month", "quantity", "amount", "discount_pct", "shipping_cost", "sku_rating"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

assembler = VectorAssembler(
    inputCols=[f"{c}_idx" for c in cat_cols] + num_cols,
    outputCol="features"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    maxBins=256,   # Must be >= max distinct categories in sku_id / delivery_partner_id
    seed=42
)

pipeline = Pipeline(stages=indexers + [assembler, rf])

# 4. Train / Test split and evaluation
train_df, test_df = historical_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_df)

test_preds = model.transform(test_df)

auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(test_preds)
f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(test_preds)
print(f"Test ROC-AUC: {auc:.4f} | Test F1-Score: {f1:.4f}")

# 5. Inspect top drivers of delivery delays (Feature Importances)
rf_model = model.stages[-1]
feature_names = [f"{c}_idx" for c in cat_cols] + num_cols
importances = sorted(zip(feature_names, rf_model.featureImportances), key=lambda x: x[1], reverse=True)

print("\nTop Delay Drivers:")
for feat, score in importances:
    print(f"  {feat:<25}: {score:.4f}")

# 6. Score open/in-transit orders for delay risk (or score test_preds if all 2025 orders are delivered)
target_to_score = open_orders_df if open_orders_df.count() > 0 else test_df

risk_scored_df = (
    model.transform(target_to_score)
    .withColumn("delay_probability", F.round(vector_to_array(F.col("probability"))[1], 4))
    .withColumn(
        "delay_risk_tier",
        F.when(F.col("delay_probability") >= 0.70, "High Risk")
         .when(F.col("delay_probability") >= 0.40, "Medium Risk")
         .otherwise("Low Risk")
    )
    .select(
        "order_id",
        "order_date",
        "promised_date",
        "warehouse_id",
        "delivery_partner_id",
        "sku_id",
        "delay_probability",
        "delay_risk_tier"
    )
    .orderBy(F.col("delay_probability").desc())
)

risk_scored_df.show(20, truncate=False)


In [ ]:
risk_scored_df.write.mode('append')\
                    .saveAsTable('dbt_adv_pr.gold.deliver_at_risk')

In [ ]:
orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
wh_df = spark.read.table("dbt_adv_pr.silver.warehouse")
cust_df = spark.read.table("dbt_adv_pr.silver.customer")

# 1. Join orders with warehouse origin and customer destination
routes_raw = (
    orders_df
    .filter(F.col("delivered_date").isNotNull() & F.col("shipping_cost").isNotNull())
    .join(
        wh_df.select(
            "warehouse_id",
            "warehouse_name",
            F.col("location").alias("origin_location")
        ),
        on="warehouse_id",
        how="left"
    )
    .join(
        cust_df.select(
            "customer_id",
            F.col("location").alias("dest_location")
        ),
        on="customer_id",
        how="left"
    )
    .withColumn("total_delivery_days", F.datediff(F.col("delivered_date"), F.col("order_date")))
    .withColumn("transit_days", F.datediff(F.col("delivered_date"), F.col("dispatch_date")))
)

# 2. Aggregate Cost, Speed, and On-Time metrics per route
route_metrics = (
    routes_raw
    .groupBy("warehouse_id", "warehouse_name", "origin_location", "dest_location")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.avg("shipping_cost"), 2).alias("avg_shipping_cost"),
        F.round(F.sum("shipping_cost") / F.sum("quantity"), 2).alias("cost_per_unit"),
        F.round(F.avg("total_delivery_days"), 2).alias("avg_delivery_days"),
        F.round(F.avg("transit_days"), 2).alias("avg_transit_days"),
        F.round(
            F.avg(F.when(F.col("delivered_date") <= F.col("promised_date"), 100.0).otherwise(0.0)),
            1
        ).alias("on_time_pct")
    )
    .filter(F.col("total_orders") >= 10)  # Filter out rare one-off routes; adjust as needed
)

# 3. Min-Max Normalize Cost & Speed onto a 0-100 scale (lower cost & fewer days = higher score)
w_all = w.rowsBetween(w.unboundedPreceding, w.unboundedFollowing)

norm_routes = (
    route_metrics
    .withColumn("min_cost", F.min("avg_shipping_cost").over(w_all))
    .withColumn("max_cost", F.max("avg_shipping_cost").over(w_all))
    .withColumn("min_days", F.min("avg_delivery_days").over(w_all))
    .withColumn("max_days", F.max("avg_delivery_days").over(w_all))
    .withColumn(
        "cost_score",
        F.when(F.col("max_cost") == F.col("min_cost"), F.lit(100.0))
         .otherwise(((F.col("max_cost") - F.col("avg_shipping_cost")) / (F.col("max_cost") - F.col("min_cost"))) * 100)
    )
    .withColumn(
        "speed_score",
        F.when(F.col("max_days") == F.col("min_days"), F.lit(100.0))
         .otherwise(((F.col("max_days") - F.col("avg_delivery_days")) / (F.col("max_days") - F.col("min_days"))) * 100)
    )
)

# 4. Compute balanced Cost vs. Speed Score (50% Cost, 50% Speed) and rank routes
w_cost, w_speed = 0.50, 0.50

best_routes_df = (
    norm_routes
    .withColumn(
        "cost_speed_score",
        F.round((F.col("cost_score") * F.lit(w_cost)) + (F.col("speed_score") * F.lit(w_speed)), 2)
    )
    .withColumn("rank", F.dense_rank().over(w.orderBy(F.col("cost_speed_score").desc())))
    .select(
        "rank",
        "warehouse_id",
        "warehouse_name",
        "origin_location",
        "dest_location",
        "total_orders",
        "avg_shipping_cost",
        "cost_per_unit",
        "avg_delivery_days",
        "on_time_pct",
        "cost_speed_score"
    )
    .orderBy("rank")
)

# Top 20 most efficient routes (best balance of low cost and fast delivery)
best_routes_df.show(20, truncate=False)

# Bottom 10 worst-performing routes (expensive and slow)
best_routes_df.orderBy(F.col("cost_speed_score").asc()).show(10, truncate=False)


In [ ]:
best_routes_df = best_routes_df.dropna()

In [ ]:
best_routes_df.write.mode('append')\
                    .saveAsTable("dbt_adv_pr.gold.delivery_best_routes")

***Service Level Agreement***

In [ ]:
orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
partner_df = spark.read.table("dbt_adv_pr.silver.delivery_partner")

delivered_orders = (
    orders_df
    .filter(F.col("delivered_date").isNotNull() & F.col("promised_date").isNotNull())
    .withColumn(
        "is_sla_breached",
        F.when(F.col("delivered_date") > F.col("promised_date"), 1).otherwise(0)
    )
    .withColumn(
        "days_late",
        F.when(
            F.col("delivered_date") > F.col("promised_date"),
            F.datediff(F.col("delivered_date"), F.col("promised_date"))
        )
    )
)

sla_metrics = (
    delivered_orders
    .groupBy("delivery_partner_id")
    .agg(
        F.countDistinct("order_id").alias("delivered_orders"),
        F.sum("is_sla_breached").alias("breached_orders"),
        F.round((F.sum("is_sla_breached") / F.countDistinct("order_id")) * 100, 2).alias("sla_breach_rate_pct"),
        F.round((1 - (F.sum("is_sla_breached") / F.countDistinct("order_id"))) * 100, 2).alias("on_time_sla_pct"),
        F.round(F.avg("days_late"), 2).alias("avg_days_late_when_breached"),
        F.round(F.avg("shipping_cost"), 2).alias("avg_shipping_cost")
    )
)

partner_sla_df = (
    partner_df
    .join(sla_metrics, on="delivery_partner_id", how="right")
    .withColumn("breach_rank", F.dense_rank().over(w.orderBy(F.col("sla_breach_rate_pct").desc())))
    .orderBy(F.col("sla_breach_rate_pct").desc())
)

partner_sla_df.show(truncate=False)


In [ ]:
partner_sla_df = partner_sla_df.dropna()
partner_sla_df.show(20, truncate=False)

In [ ]:
partner_sla_df.write.mode('append')\
                .saveAsTable('dbt_adv_pr.gold.partner_sla')